In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import zipfile
import shutil
import os
import re

ROOT = Path("/content/drive/MyDrive/Dhaka Smart Waste V4.0 2025")
ZIP_DIR = ROOT / "PARQUET_PARTS"

WORK_DIR = Path("/content/parquet_validation")

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

WORK_DIR.mkdir(parents=True)

print("ZIP folder:", ZIP_DIR)
print("Exists:", ZIP_DIR.exists())

Mounted at /content/drive
ZIP folder: /content/drive/MyDrive/Dhaka Smart Waste V4.0 2025/PARQUET_PARTS
Exists: True


In [3]:
zip_files = sorted(
    ZIP_DIR.glob("dhaka_smart_waste_v4_0_PARQUET_PART_*_of_23.zip")
)

print("ZIP files found:", len(zip_files))

for z in zip_files:
    print(z.name)

assert len(zip_files) == 23, \
    f"ERROR: Expected 23 ZIPs, found {len(zip_files)}"

print("\nExtracting...")

for i, zpath in enumerate(zip_files, 1):

    print(f"{i:02d}/23  {zpath.name}")

    try:
        with zipfile.ZipFile(zpath, "r") as z:

            # ZIP integrity test
            bad_member = z.testzip()

            if bad_member is not None:
                raise RuntimeError(
                    f"Corrupt member: {bad_member}"
                )

            z.extractall(WORK_DIR)

    except Exception as e:
        print("\n❌ ZIP ERROR:", zpath.name)
        print(e)
        raise

print("\n✅ All 23 ZIP archives extracted successfully.")

ZIP files found: 23
dhaka_smart_waste_v4_0_PARQUET_PART_01_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_02_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_03_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_04_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_05_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_06_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_07_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_08_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_09_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_10_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_11_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_12_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_13_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_14_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_15_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_16_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_17_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_18_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_19_of_23.zip
dhaka_smart_waste_v4_0_PARQUET_PART_20_of_23.zip


In [4]:
!pip -q install pyarrow

import pyarrow.parquet as pq

parquet_files = sorted(
    WORK_DIR.rglob("observations_part_*.parquet")
)

print("Observation Parquet files found:", len(parquet_files))

results = []
bad_files = []

for p in parquet_files:

    try:
        pf = pq.ParquetFile(p)

        rows = pf.metadata.num_rows
        cols = pf.metadata.num_columns
        row_groups = pf.metadata.num_row_groups

        # Force schema/footer read
        schema = pf.schema_arrow

        results.append({
            "file": p.name,
            "rows": rows,
            "columns": cols,
            "row_groups": row_groups,
            "status": "OK"
        })

    except Exception as e:

        bad_files.append({
            "file": p.name,
            "error": str(e)
        })

        results.append({
            "file": p.name,
            "rows": None,
            "columns": None,
            "row_groups": None,
            "status": "CORRUPT"
        })


print("\nTotal Parquet files:", len(results))
print("Readable:", sum(x["status"] == "OK" for x in results))
print("Bad:", len(bad_files))

if bad_files:
    print("\n❌ BAD FILES:")
    for x in bad_files:
        print(x["file"])
        print("   ", x["error"][:300])

else:
    print("\n✅ Every Parquet file is readable.")

Observation Parquet files found: 92

Total Parquet files: 92
Readable: 86
Bad: 6

❌ BAD FILES:
observations_part_031.parquet
    Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
observations_part_034.parquet
    Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
observations_part_035.parquet
    Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
observations_part_036.parquet
    Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
observations_part_037.parquet
    Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
observations_part_038.parquet
    Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.


In [5]:
EXPECTED_FILES = 92
EXPECTED_TOTAL_ROWS = 43_800_000

errors = []
total_rows = 0

files_by_index = {}

for p in parquet_files:

    m = re.search(
        r"observations_part_(\d+)\.parquet",
        p.name
    )

    if m:
        idx = int(m.group(1))
        files_by_index[idx] = p


# Check all expected indexes exist
for i in range(EXPECTED_FILES):

    if i not in files_by_index:
        errors.append(
            f"MISSING: observations_part_{i:03d}.parquet"
        )
        continue

    p = files_by_index[i]

    try:
        rows = pq.ParquetFile(p).metadata.num_rows

    except Exception as e:
        errors.append(
            f"CORRUPT: {p.name} -> {e}"
        )
        continue


    expected_rows = (
        120_000 if i == 91
        else 480_000
    )

    if rows != expected_rows:
        errors.append(
            f"WRONG ROW COUNT: {p.name} "
            f"has {rows:,}, expected {expected_rows:,}"
        )

    total_rows += rows


print("=" * 60)
print("Parquet files :", len(files_by_index))
print("Total rows    :", f"{total_rows:,}")
print("Expected rows :", f"{EXPECTED_TOTAL_ROWS:,}")
print("Errors        :", len(errors))
print("=" * 60)


if errors:

    print("\n❌ VALIDATION FAILED\n")

    for e in errors:
        print(e)

else:

    assert len(files_by_index) == 92
    assert total_rows == 43_800_000

    print("\n✅ FULL DATASET VALIDATED")
    print("✅ 92 observation Parquet files")
    print("✅ 43,800,000 total rows")
    print("✅ All expected row counts correct")
    print("✅ All Parquet files readable")

Parquet files : 92
Total rows    : 40,920,000
Expected rows : 43,800,000
Errors        : 6

❌ VALIDATION FAILED

CORRUPT: observations_part_031.parquet -> Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
CORRUPT: observations_part_034.parquet -> Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
CORRUPT: observations_part_035.parquet -> Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
CORRUPT: observations_part_036.parquet -> Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
CORRUPT: observations_part_037.parquet -> Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
CORRUPT: observations_part_038.parquet -> Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
